# Name That LLM (simple version)

**Goal:** given one passage of text, guess which of 18 LLMs wrote it.

**The catch:** the training passages come from about 150 prompts, and every model answered every prompt once.
The test set uses *new* prompts, so the topic can't help us. Only writing style can.

**Plan**
1. Find which passages answer the same prompt (cluster by topic)
2. Turn each passage into style numbers
3. Subtract each prompt's average, which removes the topic and keeps the style
4. Train a classifier and test it on prompts it has never seen
5. Use the rule "each model answers each prompt once" to clean up the guesses
6. Let Optuna choose the settings, train the final model, and score it on prompts held out until the very end

*This notebook is the explanation and the training. It does not call Airflow, Docker or MLflow itself: those run the
notebook and pick up the files it saves in `../models` (see the README).*

In [1]:
import re
from collections import Counter

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score, balanced_accuracy_score
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, normalize

TRAIN_FILE = "../data/train.csv"

## 1. Load the data

In [2]:
df = pd.read_csv(TRAIN_FILE).dropna(subset=["text", "model"]).reset_index(drop=True)

n_models = df["model"].nunique()
n_prompts = round(len(df) / n_models)   # each model answers each prompt once

print(len(df), "passages,", n_models, "models, about", n_prompts, "prompts")

2699 passages, 18 models, about 150 prompts


## 2. Find the hidden prompt groups

`train.csv` has a `prompt_id` column, but the test set won't. So we rebuild the groups ourselves:
turn each passage into topic numbers, then cluster. We keep only rare content words (dropping common ones)
so that passages cluster by *subject*, not by author.

In [3]:
# word weights (TF-IDF) using only rare, topical words
vectorizer = TfidfVectorizer(
    stop_words="english",
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
    sublinear_tf=True,
    min_df=2,
    max_df=0.05,
)
word_weights = vectorizer.fit_transform(df["text"])

# squash to 200 numbers per passage (this step is called LSA)
topic_vectors = TruncatedSVD(200, random_state=42).fit_transform(word_weights)
topic_vectors = normalize(topic_vectors)

# merge the closest passages together until we have one cluster per prompt
clusterer = AgglomerativeClustering(n_clusters=n_prompts, metric="cosine", linkage="average")
df["group"] = clusterer.fit_predict(topic_vectors)

**Did it work?** A good cluster has about 18 passages and never the same model twice.
Since this file has the real `prompt_id`, we can also compare against the truth.

In [4]:
sizes = df.groupby("group").size()
has_repeat = df.groupby("group")["model"].apply(lambda models: models.duplicated().any())

print("median cluster size:", sizes.median())
print("clusters with a repeated model: %.1f%%" % (100 * has_repeat.mean()))
print("agreement with real prompts (1.0 = perfect):", round(adjusted_rand_score(df["prompt_id"], df["group"]), 3))

median cluster size: 18.0
clusters with a repeated model: 2.7%
agreement with real prompts (1.0 = perfect): 0.977


## 3. Why we must split by prompt

If we split passages at random, the other 17 answers to the same prompt end up in training with *other* labels,
which confuses the model. Splitting by prompt copies the real test situation.
Here is the same simple model scored both ways.

In [5]:
simple_model = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2, max_features=100_000),
    LogisticRegression(C=5, class_weight="balanced", max_iter=1000),
)

random_split = StratifiedKFold(5, shuffle=True, random_state=42)
random_pred = cross_val_predict(simple_model, df["text"], df["model"], cv=random_split, n_jobs=-1)

prompt_split = GroupKFold(5)
prompt_pred = cross_val_predict(simple_model, df["text"], df["model"], cv=prompt_split, groups=df["group"], n_jobs=-1)

print("random split :", round(balanced_accuracy_score(df["model"], random_pred), 3))
print("split by prompt:", round(balanced_accuracy_score(df["model"], prompt_pred), 3), "(this is the honest number)")
print("guessing at random would give", round(1 / n_models, 3))

random split : 0.114
split by prompt: 0.313 (this is the honest number)
guessing at random would give 0.056


## 4. Style features

Three views of how a passage is written:

* **characters** (2 to 5 letters in a row): spelling and punctuation habits
* **words** (single words and pairs): phrasing
* **skeleton**: keep the 150 most common words and punctuation, replace every other word with `W`.
  `The cat sat; it was warm.` becomes `the W W ; it was W .` This captures sentence rhythm and has almost no topic in it.

In [6]:
def make_skeleton(text, common_words):
    pieces = []
    for token in re.findall(r"[A-Za-z']+|[^\s A-Za-z']", text):
        lower = token.lower()
        if lower in common_words:
            pieces.append(lower)
        elif token[0].isalpha() or token[0] == "'":
            pieces.append("W")
        else:
            pieces.append(token)   # punctuation stays as it is
    return " ".join(pieces)


def fit_features(train_text, use_skeleton=True):
    # Everything is learned from the training text only.
    all_words = [w.lower() for t in train_text for w in re.findall(r"[A-Za-z']+", t)]
    common_words = {word for word, count in Counter(all_words).most_common(150)}

    vectorizers = {
        "chars": TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2, max_features=20000, dtype=np.float32),
        "words": TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=15000, dtype=np.float32),
    }
    if use_skeleton:
        vectorizers["skeleton"] = TfidfVectorizer(ngram_range=(1, 4), token_pattern=r"\S+", lowercase=False, sublinear_tf=True,
                                                  min_df=3, max_features=20000, dtype=np.float32)

    for name, vec in vectorizers.items():
        vec.fit(skeletons(train_text, common_words) if name == "skeleton" else train_text)
    return {"common_words": common_words, "vectorizers": vectorizers}


def skeletons(texts, common_words):
    return [make_skeleton(t, common_words) for t in texts]


def transform_features(features, texts):
    blocks = []
    for name, vec in features["vectorizers"].items():
        docs = skeletons(texts, features["common_words"]) if name == "skeleton" else texts
        blocks.append(vec.transform(docs).toarray())
    return np.hstack(blocks)


def build_features(train_text, test_text, use_skeleton=True):
    features = fit_features(train_text, use_skeleton)
    return transform_features(features, train_text), transform_features(features, test_text)

## 5. Remove the topic, then fix the guesses

**Centering:** inside each prompt group all 18 answers share the topic. Subtracting the group's average
cancels it out and leaves what is different about each author.

**Assignment:** each model answers each prompt exactly once, so inside a group the 18 passages should get 18 *different*
models. `linear_sum_assignment` finds the best one-to-one matching using the classifier's probabilities.

In [7]:
def center_by_group(features, groups):
    centered = features.copy()
    for g in np.unique(groups):
        rows = (groups == g)
        centered[rows] = features[rows] - features[rows].mean(axis=0)
    return centered


def assign_one_model_each(probs, groups):
    best_guess = probs.argmax(axis=1)
    for g in np.unique(groups):
        rows = np.where(groups == g)[0]
        cost = -np.log(probs[rows] + 1e-9)          # low cost = likely
        if len(rows) > n_models:                     # a merged cluster: let models repeat
            cost = np.tile(cost, int(np.ceil(len(rows) / n_models)))
        row_ids, model_ids = linear_sum_assignment(cost)
        best_guess[rows[row_ids]] = model_ids % n_models
    return best_guess

## 6. Put it together and test it

Every score below comes from prompts the model has never seen (5 folds).
We run it twice, without and with the skeleton features, so you can see what they add.
This cell takes a few minutes.

In [8]:
encoder = LabelEncoder().fit(df["model"])
y = encoder.transform(df["model"])
groups = df["group"].to_numpy()

scores = {}   # (features, step) -> list of scores, one per fold

for use_skeleton in [False, True]:
    name = "with skeleton" if use_skeleton else "without skeleton"

    for fold, (train_rows, test_rows) in enumerate(GroupKFold(5).split(df, y, groups)):
        train_x, test_x = build_features(df["text"].iloc[train_rows], df["text"].iloc[test_rows], use_skeleton)

        train_x = center_by_group(train_x, groups[train_rows])
        test_x = center_by_group(test_x, groups[test_rows])

        classifier = LogisticRegression(C=5, class_weight="balanced", max_iter=600)
        classifier.fit(train_x, y[train_rows])
        probs = classifier.predict_proba(test_x)

        plain_guess = probs.argmax(axis=1)
        fixed_guess = assign_one_model_each(probs, groups[test_rows])

        scores.setdefault((name, "centered"), []).append(balanced_accuracy_score(y[test_rows], plain_guess))
        scores.setdefault((name, "centered + assignment"), []).append(balanced_accuracy_score(y[test_rows], fixed_guess))
        print(name, "fold", fold + 1, "done")

results = pd.DataFrame({key: [np.mean(vals), np.std(vals)] for key, vals in scores.items()}, index=["mean", "std"]).T
results.round(3)

without skeleton fold 1 done


without skeleton fold 2 done


without skeleton fold 3 done


without skeleton fold 4 done


without skeleton fold 5 done


with skeleton fold 1 done


with skeleton fold 2 done


with skeleton fold 3 done


with skeleton fold 4 done


with skeleton fold 5 done


mean    std
without skeleton centered               0.381  0.013
                 centered + assignment  0.422  0.010
with skeleton    centered               0.431  0.013
                 centered + assignment  0.466  0.020

## 7. Ensemble: add a language model's opinion

A second, different kind of model can fix some of the TF-IDF model's mistakes.
We turn every passage into 768 numbers with DistilBERT (a small pretrained language model),
train a second classifier on those, and blend the two sets of probabilities.

The language model gets only a **small weight (10%)**. On its own it is weaker than TF-IDF, and at 30% it starts to hurt.
In our experiments 10% and 20% both helped in all 5 folds.
Those weights were picked on these same folds, so treat the gain as slightly optimistic.

Embedding the passages needs PyTorch + `sentence-transformers` and takes a few minutes on CPU.
The result is saved to `../data/embeddings/distilbert.npy`, so it only runs once.

In [9]:
from pathlib import Path
from sklearn.preprocessing import StandardScaler

EMBEDDING_FILE = Path("../data/embeddings/distilbert.npy")

if EMBEDDING_FILE.exists():
    embeddings = np.load(EMBEDDING_FILE)
else:
    from sentence_transformers import SentenceTransformer
    language_model = SentenceTransformer("distilbert-base-uncased-finetuned-sst-2-english", device="cpu")
    language_model.max_seq_length = 512
    embeddings = language_model.encode(df["text"].tolist(), batch_size=16, normalize_embeddings=True)
    EMBEDDING_FILE.parent.mkdir(exist_ok=True)
    np.save(EMBEDDING_FILE, embeddings)

print(embeddings.shape)

(2699, 768)


In [10]:
def blend(probs_main, probs_other, other_weight):
    # weighted geometric mean: average the log-probabilities, then turn them back into probabilities
    logs = (1 - other_weight) * np.log(probs_main + 1e-9) + other_weight * np.log(probs_other + 1e-9)
    mixed = np.exp(logs)
    return mixed / mixed.sum(axis=1, keepdims=True)


LANGUAGE_MODEL_WEIGHT = 0.1
ensemble_scores = {"TF-IDF only": [], "TF-IDF + 10% DistilBERT": []}

for fold, (train_rows, test_rows) in enumerate(GroupKFold(5).split(df, y, groups)):
    # model 1: the TF-IDF style features (C=1 worked a bit better than 5)
    train_x, test_x = build_features(df["text"].iloc[train_rows], df["text"].iloc[test_rows])
    train_x = center_by_group(train_x, groups[train_rows])
    test_x = center_by_group(test_x, groups[test_rows])
    tfidf_probs = LogisticRegression(C=1, class_weight="balanced", max_iter=600).fit(train_x, y[train_rows]).predict_proba(test_x)

    # model 2: DistilBERT numbers, centred by prompt and standardised
    train_e = center_by_group(embeddings[train_rows], groups[train_rows])
    test_e = center_by_group(embeddings[test_rows], groups[test_rows])
    scaler = StandardScaler().fit(train_e)
    lm_probs = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000).fit(scaler.transform(train_e), y[train_rows]).predict_proba(scaler.transform(test_e))

    mixed_probs = blend(tfidf_probs, lm_probs, LANGUAGE_MODEL_WEIGHT)
    for name, probs in [("TF-IDF only", tfidf_probs), ("TF-IDF + 10% DistilBERT", mixed_probs)]:
        guess = assign_one_model_each(probs, groups[test_rows])
        ensemble_scores[name].append(balanced_accuracy_score(y[test_rows], guess))
    print("fold", fold + 1, "done")

pd.DataFrame({k: [np.mean(v), np.std(v)] for k, v in ensemble_scores.items()}, index=["mean", "std"]).T.round(3)

fold 1 done


fold 2 done


fold 3 done


fold 4 done


fold 5 done


,mean,std
TF-IDF only,0.477,0.025
TF-IDF + 10% DistilBERT,0.521,0.018


## 8. Hold out some prompts, then let Optuna choose the settings

So far every score came from cross-validation over all the prompts, with settings (C=1, C=0.1, a 10% blend) picked by hand
on those same folds. Now we do it properly:

* **Hold out 20% of the prompts.** They are not used for training or for choosing settings. They are the final exam, and
  later they are also the clean reference data for monitoring the deployed model.
* **Optuna searches three settings** on the other 80% of the prompts, using the same 5-fold split by prompt:
  `style_c` (how strongly the style classifier is regularised), `embedding_c` (the same for the DistilBERT classifier) and
  `embedding_weight` (how much the DistilBERT opinion counts in the blend). Its first try is the hand-picked settings, so
  it can only end up equal or better on these folds.
* Fitting a model is the slow part, so the two C settings are limited to three values each. Every value is fitted once per
  fold, and Optuna then picks the best combination from those saved results. The blend weight can be anything from 0 to 0.4.

Optuna's score is slightly optimistic, because it picks the best of 20 tries on the same folds. The held-out prompts in
section 9 give the honest number. Expect the gain over the hand-picked settings to be small: in earlier experiments, tuning
did not help once it was scored honestly.

In [11]:
import datetime
import hashlib
import json

import joblib
import optuna
from sklearn.model_selection import GroupShuffleSplit

MODEL_VERSION = "0.2.0"
MODEL_DIR = Path("../models")

N_TRIALS = 20
STYLE_C_CHOICES = [0.3, 1, 3]
EMBEDDING_C_CHOICES = [0.03, 0.1, 0.3]

# hold out 20% of the prompts (whole groups); the same ones every run
fit_rows, holdout_rows = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(df, y, groups))
print(len(fit_rows), "passages to train on,", len(holdout_rows), "held out")

# 5 folds inside the training part, again by prompt
folds = list(GroupKFold(5).split(fit_rows, groups=groups[fit_rows]))

2171 passages to train on, 528 held out


/home/airflow/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# For every candidate C and every fold: fit once and keep the predicted probabilities. This is the slow part.
style_probs, embedding_probs = {}, {}

for fold, (a, b) in enumerate(folds):
    rows_a, rows_b = fit_rows[a], fit_rows[b]

    train_x, test_x = build_features(df["text"].iloc[rows_a], df["text"].iloc[rows_b])
    train_x, test_x = center_by_group(train_x, groups[rows_a]), center_by_group(test_x, groups[rows_b])
    for c in STYLE_C_CHOICES:
        classifier = LogisticRegression(C=c, class_weight="balanced", max_iter=600).fit(train_x, y[rows_a])
        style_probs[c, fold] = classifier.predict_proba(test_x)

    train_e, test_e = center_by_group(embeddings[rows_a], groups[rows_a]), center_by_group(embeddings[rows_b], groups[rows_b])
    scaler = StandardScaler().fit(train_e)
    for c in EMBEDDING_C_CHOICES:
        classifier = LogisticRegression(C=c, class_weight="balanced", max_iter=2000).fit(scaler.transform(train_e), y[rows_a])
        embedding_probs[c, fold] = classifier.predict_proba(scaler.transform(test_e))
    print("fold", fold + 1, "done")

fold 1 done


fold 2 done


fold 3 done


fold 4 done


fold 5 done


In [13]:
optuna.logging.set_verbosity(optuna.logging.WARNING)


def cv_score(style_c, embedding_c, embedding_weight):
    scores = []
    for fold, (a, b) in enumerate(folds):
        rows_b = fit_rows[b]
        probs = blend(style_probs[style_c, fold], embedding_probs[embedding_c, fold], embedding_weight)
        guess = assign_one_model_each(probs, groups[rows_b])
        scores.append(balanced_accuracy_score(y[rows_b], guess))
    return np.mean(scores)


def objective(trial):
    return cv_score(
        trial.suggest_categorical("style_c", STYLE_C_CHOICES),
        trial.suggest_categorical("embedding_c", EMBEDDING_C_CHOICES),
        trial.suggest_float("embedding_weight", 0.0, 0.4),
    )


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.enqueue_trial({"style_c": 1, "embedding_c": 0.1, "embedding_weight": 0.1})   # start from the hand-picked settings
study.optimize(objective, n_trials=N_TRIALS)

best = study.best_params
hand_picked = study.trials[0].value
print("hand-picked settings (C=1, C=0.1, weight 0.1):", round(hand_picked, 3))
print("Optuna's best:", {name: round(value, 3) for name, value in best.items()}, "->", round(study.best_value, 3))
print("gain over the hand-picked settings:", round(study.best_value - hand_picked, 3), "(a gain this small is within the noise)" if study.best_value - hand_picked < 0.01 else "")

trial_table = study.trials_dataframe()[["number", "value", "params_style_c", "params_embedding_c", "params_embedding_weight"]]
trial_table.sort_values("value", ascending=False).head(5).round(3)

hand-picked settings (C=1, C=0.1, weight 0.1): 0.517
Optuna's best: {'style_c': 1, 'embedding_c': 0.1, 'embedding_weight': 0.1} -> 0.517
gain over the hand-picked settings: 0.0 (a gain this small is within the noise)


,number,value,params_style_c,params_embedding_c,params_embedding_weight
0,0,0.517,1.0,0.10,0.100
3,3,0.512,3.0,0.03,0.245
15,15,0.509,1.0,0.03,0.155
12,12,0.509,3.0,0.03,0.134
18,18,0.508,3.0,0.10,0.164


## 9. Train the final model, test it on the held-out prompts, and save a version

Train on the 80% training prompts with the settings Optuna chose, then score the model on the held-out 20%. That score is
the honest one: those prompts were not used for training or for choosing any setting. The model is trained on 80% of the
data on purpose, so that the held-out prompts stay a real test for the deployed service too.

Four files are written to `../models`:

* `model.joblib`: the model. It holds only standard scikit-learn objects and plain Python, so it loads without this notebook.
* `model_info.json`: the version, the held-out score, the cross-validated score, the settings and a hash of `train.csv`,
  so every model says which data it was trained on.
* `holdout.csv`: the held-out passages, with their true LLM. The service is checked against them, and they are the baseline
  for monitoring.
* `optuna_trials.csv`: every setting Optuna tried and its score.

**Version numbers** look like `0.2.0`. Raise `MODEL_VERSION` when the method changes (new features, a different classifier).
When the model is registered in MLflow (README), the last number goes up by itself for every retrain.

In [14]:
# Train on the training prompts only, with the settings Optuna chose.
features = fit_features(df["text"].iloc[fit_rows])
style_x = center_by_group(transform_features(features, df["text"].iloc[fit_rows]), groups[fit_rows])
style_classifier = LogisticRegression(C=best["style_c"], class_weight="balanced", max_iter=600).fit(style_x, y[fit_rows])

embedding_x = center_by_group(embeddings[fit_rows], groups[fit_rows])
embedding_scaler = StandardScaler().fit(embedding_x)
embedding_classifier = LogisticRegression(C=best["embedding_c"], class_weight="balanced", max_iter=2000).fit(
    embedding_scaler.transform(embedding_x), y[fit_rows])

model = {
    "classes": encoder.classes_,
    "features": features,
    "style_classifier": style_classifier,
    "embedding_scaler": embedding_scaler,
    "embedding_classifier": embedding_classifier,
    "embedding_weight": best["embedding_weight"],
}
print("trained on", len(fit_rows), "passages")

trained on 2171 passages


In [15]:
def predict_with(saved, texts, embeddings, groups):
    style_x = center_by_group(transform_features(saved["features"], texts), groups)
    style_probs = saved["style_classifier"].predict_proba(style_x)

    embedding_x = saved["embedding_scaler"].transform(center_by_group(embeddings, groups))
    embedding_probs = saved["embedding_classifier"].predict_proba(embedding_x)

    probs = blend(style_probs, embedding_probs, saved["embedding_weight"])
    return saved["classes"][assign_one_model_each(probs, groups)]


MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(model, MODEL_DIR / "model.joblib")
print("saved", MODEL_DIR / "model.joblib")

saved ../models/model.joblib


### Score the saved model on the held-out prompts

Load the file the way someone else would, and use it on the held-out passages. Passages have to be grouped by prompt first,
because centering and assignment need the other passages from the same prompt; here we use the groups from section 2.
**This is the honest score of the model.** The Optuna score in section 8 is a little optimistic, and the scores in sections
6 and 7 were measured with settings picked on the same folds.

In [16]:
loaded = joblib.load(MODEL_DIR / "model.joblib")

holdout_guess = predict_with(loaded, df["text"].iloc[holdout_rows], embeddings[holdout_rows], groups[holdout_rows])
holdout_score = balanced_accuracy_score(df["model"].iloc[holdout_rows], holdout_guess)

print(len(holdout_rows), "held-out passages from", len(np.unique(groups[holdout_rows])), "prompts")
print("balanced accuracy on the held-out prompts:", round(holdout_score, 3), "(guessing would give", round(1 / n_models, 3), ")")

528 held-out passages from 30 prompts
balanced accuracy on the held-out prompts: 0.478 (guessing would give 0.056 )


### Write down what this version is

The version number, the held-out score, the settings and the hash of the training data go into `model_info.json`. The
held-out passages and Optuna's trials are saved next to it.

In [17]:
info_file = MODEL_DIR / "model_info.json"
previous = json.loads(info_file.read_text()) if info_file.exists() else None

info = {
    "version": MODEL_VERSION,
    "trained_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "balanced_accuracy": round(float(holdout_score), 3),
    "how_scored": "held-out prompts: 20% of the prompts, never used for training or for choosing the settings; centering + assignment",
    "cv_balanced_accuracy": round(float(study.best_value), 3),
    "settings": {
        "style_c": best["style_c"],
        "embedding_c": best["embedding_c"],
        "embedding_weight": round(best["embedding_weight"], 4),
    },
    "optuna_trials": N_TRIALS,
    "training_passages": len(fit_rows),
    "holdout_passages": len(holdout_rows),
    "train_csv_md5": hashlib.md5(Path(TRAIN_FILE).read_bytes()).hexdigest(),   # the same hash DVC records for train.csv
}

if previous is not None:
    print("previous file: version", previous["version"], "scored", previous["balanced_accuracy"])
    print("this version : version", info["version"], "scored", info["balanced_accuracy"])
    if previous["version"] == info["version"]:
        print("same version number as before: raise MODEL_VERSION if the method changed")

info_file.write_text(json.dumps(info, indent=2))
pd.DataFrame({
    "text": df["text"].iloc[holdout_rows].to_numpy(),
    "model": df["model"].iloc[holdout_rows].to_numpy(),
    "group": groups[holdout_rows],
}).to_csv(MODEL_DIR / "holdout.csv", index=False)
trial_table.to_csv(MODEL_DIR / "optuna_trials.csv", index=False)

# JupyterLab, Airflow and the API run as different users in their containers. Let all of them overwrite these files on a later run.
for name in ["model.joblib", "model_info.json", "holdout.csv", "optuna_trials.csv"]:
    try:
        (MODEL_DIR / name).chmod(0o666)
    except PermissionError:
        pass   # the file belongs to someone else; they already opened it up

print("saved model_info.json, holdout.csv and optuna_trials.csv in", MODEL_DIR)
info

previous file: version 0.2.0 scored 0.478
this version : version 0.2.0 scored 0.478
same version number as before: raise MODEL_VERSION if the method changed
saved model_info.json, holdout.csv and optuna_trials.csv in ../models


{'version': '0.2.0',
 'trained_at': '2026-11-16T03:56:33',
 'balanced_accuracy': 0.478,
 'how_scored': 'held-out prompts: 20% of the prompts, never used for training or for choosing the settings; centering + assignment',
 'cv_balanced_accuracy': 0.517,
 'settings': {'style_c': 1, 'embedding_c': 0.1, 'embedding_weight': 0.1},
 'optuna_trials': 20,
 'training_passages': 2171,
 'holdout_passages': 528,
 'train_csv_md5': '61f581c1bdd4664d588a7c23c413ba0a'}

### Version the data and the model

The notebook has now written the files in `../models`. To record this as a version, run these in a terminal from the project
folder (DVC stores the big file by its hash; git keeps the small pointer and the info):

```
dvc add models/model.joblib
git add models/model.joblib.dvc models/model_info.json models/optuna_trials.csv models/.gitignore
git commit -m "Model v0.2.0"
git tag model-v0.2.0
dvc push
```

To go back to an older version later: `git checkout model-v0.1.0 -- models/model.joblib.dvc models/model_info.json`, then
`dvc checkout models/model.joblib`. The training data is versioned the same way (`dvc add data/train.csv`, tag `data-v2`, ...).
If you run the notebook through Airflow, the model is also registered in MLflow, which numbers and promotes versions for you
(see the README).

## What to take away

* Splitting by prompt gives the honest score. Random splits mislead.
* Centering and assignment both help a lot, because both use the fact that each prompt has exactly one answer per model.
* The skeleton features add a further gain.
* Models from the same company (for example the Gemini ones) get mixed up with each other most.
* Blending in a small share of a language model (DistilBERT) adds a further gain; averaging more TF-IDF models does not.
* Optuna picks the settings on the training prompts; the held-out prompts then give a score that was not tuned on.
* At test time the same recipe needs the test passages grouped by prompt too.
* Data and models are versioned (DVC and git tags), and each saved model records its version, its score and the hash of the data it was trained on.